<a href="https://colab.research.google.com/github/ZzwahwahzZ/Conquer-Mobile-Assets./blob/main/notebooks/Conquer_Mobile_UV_Texture_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Conquer Mobile — اختبار UV Texture على 3 مجسمات

دفتر مستقل عن اختبار Vertex Color السابق. ينشئ UV atlas باستخدام xatlas، ويخبز إسقاط الصورة داخل الخامة، ثم يصدر GLB وPNG وتقريرًا وZIP ويحفظ نسخة على Google Drive ويرفع النتائج إلى GitHub.

**لا يعيد توليد TripoSR ولا يغيّر مجلدات الأصول أو نتائج الاختبار السابق.** المطابقة صارمة: إذا لم تكن كل صورة مرتبطة بمجسم واحد فريد بالاسم، يتوقف الدفتر بدل استخدام ترتيب أبجدي تخميني.

قبل التشغيل، أضف GitHub Personal Access Token في Colab Secrets بالاسم `GITHUB_TOKEN` مع صلاحية Contents: Read and write وفعّل Notebook access. لا تضع المفتاح داخل الكود. تحتاج الجلسة إلى إنترنت وGoogle Drive.

ملاحظة فنية: هذا اختبار UV atlas بخامة إسقاط من صورة واحدة؛ ليس توليدًا دلاليًا متعدد الزوايا ولا خامة PBR إنتاجية نهائية.


In [ ]:
#@title اختبار UV Texture على 3 مجسمات — خلية واحدة
# لا يعيد توليد TripoSR، ولا يغيّر أي أصل أو نتيجة سابقة.
# ينشئ UV atlas ويخبز إسقاط الصورة داخل أطلس UV، ثم يصدر GLB + PNG + تقرير.

import os, re, json, csv, zipfile, shutil, subprocess, sys, base64
from pathlib import Path

def install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

install("trimesh>=4.4", "Pillow>=10", "numpy>=1.26", "requests>=2.31", "xatlas", "numba")

import numpy as np
import trimesh, xatlas, requests
from PIL import Image
from numba import njit
from google.colab import drive, userdata
from IPython.display import display, Markdown

print("1/8 — ربط Google Drive وتجهيز مجلد اختبار مستقل...")
drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive")
REPO = "ZzwahwahzZ/Conquer-Mobile-Assets."
REPO_URL = "https://github.com/ZzwahwahzZ/Conquer-Mobile-Assets..git"
REPO_DIR = Path("/content/Conquer-Mobile-Assets")
WORK = Path("/content/Conquer_UV_Texture_Test")
RAW = WORK / "raw_models"
OUT = WORK / "uv_texture_test"
for folder in (WORK, RAW, OUT):
    folder.mkdir(parents=True, exist_ok=True)

print("2/8 — تحديث نسخة المستودع والعثور على الأرشيف المحفوظ...")
if REPO_DIR.exists():
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
else:
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])

IMAGE_ROOT = REPO_DIR / "original_images"
if not IMAGE_ROOT.is_dir():
    raise FileNotFoundError("مجلد original_images غير موجود في المستودع.")
images = sorted(p for p in IMAGE_ROOT.rglob("*") if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"})
archives = list(DRIVE.rglob("Conquer_TripoSR_42_Models.zip"))
if not images:
    raise RuntimeError("لم يتم العثور على الصور الأصلية.")
if not archives:
    raise FileNotFoundError("لم أجد Conquer_TripoSR_42_Models.zip في Google Drive؛ لم يتم توليد أي مجسم جديد.")
MODEL_ZIP = archives[0]
with zipfile.ZipFile(MODEL_ZIP) as zf:
    members = [n for n in zf.namelist() if n.lower().endswith(".glb")]
    if not members:
        raise RuntimeError("الأرشيف لا يحتوي GLB.")
    for name in members:
        target = (RAW / name).resolve()
        if RAW.resolve() not in target.parents:
            raise RuntimeError("مسار غير آمن داخل الأرشيف: " + name)
        zf.extract(name, RAW)
meshes = sorted(p for p in RAW.rglob("*") if p.suffix.lower() == ".glb")
print(f"الصور: {len(images)} | مجسمات GLB: {len(meshes)}")

print("3/8 — مطابقة صارمة بالاسم ومعرّف الملف، بلا ترتيب أبجدي بديل...")
def norm(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())

def mesh_keys(path):
    stem = path.stem
    keys = {norm(stem)}
    # بعض ملفات التوليد تحمل اسم الفئة مرتين: characters__characters_0001...
    parts = re.split(r"__+", stem, maxsplit=1)
    if len(parts) == 2 and norm(parts[0]) and norm(parts[1]).startswith(norm(parts[0])):
        keys.add(norm(parts[1]))
    return keys

index = {}
for m in meshes:
    for key in mesh_keys(m):
        index.setdefault(key, []).append(m)

pairs, used = [], set()
unmatched, ambiguous = [], []
for img in images:
    key = norm(img.stem)
    candidates = list({str(m): m for m in index.get(key, [])}.values())
    # دعم اسم الفئة داخل المعرّف مع الاحتفاظ بتطابق الاسم الكامل.
    if not candidates:
        category = img.parent.name if img.parent != IMAGE_ROOT else ""
        candidates = list({str(m): m for m in index.get(norm(category + "_" + img.stem), [])}.values())
    candidates = [m for m in candidates if str(m) not in used]
    if len(candidates) == 1:
        pairs.append((img, candidates[0]))
        used.add(str(candidates[0]))
    elif len(candidates) == 0:
        unmatched.append(str(img.relative_to(IMAGE_ROOT)))
    else:
        ambiguous.append((str(img.relative_to(IMAGE_ROOT)), [m.name for m in candidates]))

if unmatched or ambiguous:
    raise RuntimeError(
        "توقّف آمن: المطابقة ليست كاملة وفريدة. "
        f"غير مطابق: {len(unmatched)}؛ ملتبس: {len(ambiguous)}. "
        f"أمثلة غير مطابقة: {unmatched[:5]}؛ أمثلة ملتبسة: {ambiguous[:3]}. "
        "لم أستخدم ترتيب القوائم ولم أعدّل أي أصل."
    )
if len(pairs) != len(images) or len(used) != len(images):
    raise RuntimeError("فشل تحقق واحد-إلى-واحد بين الصور والمجسمات.")
if len(pairs) < 3:
    raise RuntimeError("نحتاج 3 أزواج مؤكدة على الأقل.")

samples = pairs[:3]
for i, (im, me) in enumerate(samples, 1):
    print(f"عينة {i}: {im.relative_to(IMAGE_ROOT)}  <->  {me.name}")

print("4/8 — تجهيز دالة خبز الإسقاط داخل UV atlas...")
@njit
def bake_atlas(vertices, faces, uv, source, out_w, out_h, min_x, min_y, span_x, span_y):
    # source: RGB/RGBA image; output RGBA texture
    # Bounds are computed in regular NumPy outside Numba for Python 3.13 compatibility.
    tex = np.zeros((out_h, out_w, 4), dtype=np.uint8)
    src_h, src_w = source.shape[0], source.shape[1]

    for fi in range(faces.shape[0]):
        ia, ib, ic = faces[fi, 0], faces[fi, 1], faces[fi, 2]
        ua, ub, uc = uv[ia], uv[ib], uv[ic]
        ax, ay = ua[0] * (out_w - 1), (1.0 - ua[1]) * (out_h - 1)
        bx, by = ub[0] * (out_w - 1), (1.0 - ub[1]) * (out_h - 1)
        cx, cy = uc[0] * (out_w - 1), (1.0 - uc[1]) * (out_h - 1)
        minx = max(0, int(np.floor(min(ax, bx, cx))))
        maxx = min(out_w - 1, int(np.ceil(max(ax, bx, cx))))
        miny = max(0, int(np.floor(min(ay, by, cy))))
        maxy = min(out_h - 1, int(np.ceil(max(ay, by, cy))))
        den = (by - cy) * (ax - cx) + (cx - bx) * (ay - cy)
        if abs(den) < 1e-12:
            continue
        va, vb, vc = vertices[ia], vertices[ib], vertices[ic]
        for py in range(miny, maxy + 1):
            for px in range(minx, maxx + 1):
                x, y = px + 0.5, py + 0.5
                wa = ((by - cy) * (x - cx) + (cx - bx) * (y - cy)) / den
                wb = ((cy - ay) * (x - cx) + (ax - cx) * (y - cy)) / den
                wc = 1.0 - wa - wb
                if wa >= -1e-5 and wb >= -1e-5 and wc >= -1e-5:
                    posx = wa * va[0] + wb * vb[0] + wc * vc[0]
                    posy = wa * va[1] + wb * vb[1] + wc * vc[1]
                    su = (posx - min_x) / span_x
                    sv = (min_y + span_y - posy) / span_y
                    sx = min(src_w - 1, max(0, int(round(su * (src_w - 1)))))
                    sy = min(src_h - 1, max(0, int(round(sv * (src_h - 1)))))
                    tex[py, px, 0] = source[sy, sx, 0]
                    tex[py, px, 1] = source[sy, sx, 1]
                    tex[py, px, 2] = source[sy, sx, 2]
                    tex[py, px, 3] = source[sy, sx, 3] if source.shape[2] == 4 else 255
    return tex

def create_uv_texture(mesh_path, image_path, output_glb, output_png, resolution=1024):
    loaded = trimesh.load(mesh_path, force="mesh", process=False)
    if not isinstance(loaded, trimesh.Trimesh) or len(loaded.vertices) == 0 or len(loaded.faces) == 0:
        raise ValueError("هندسة GLB غير صالحة: " + str(mesh_path))
    mesh = loaded
    vertices = np.asarray(mesh.vertices, dtype=np.float32)
    faces = np.asarray(mesh.faces, dtype=np.uint32)

    atlas = xatlas.Atlas()
    atlas.add_mesh(vertices, faces)
    pack = xatlas.PackOptions()
    pack.resolution = resolution
    pack.padding = 4
    atlas.generate(pack_options=pack)
    vmapping, new_faces, uvs = atlas[0]
    uv_vertices = vertices[np.asarray(vmapping, dtype=np.int64)]
    new_faces = np.asarray(new_faces, dtype=np.int64).reshape((-1, 3))
    uvs = np.asarray(uvs, dtype=np.float32)

    source_img = Image.open(image_path).convert("RGBA")
    source_img.thumbnail((2048, 2048))
    source = np.asarray(source_img, dtype=np.uint8)
    # احسب الحدود خارج Numba؛ إصدارات Numba الحالية لا تدعم ndarray.min(axis=0) داخل njit.
    bounds_min = np.min(uv_vertices, axis=0).astype(np.float64)
    bounds_max = np.max(uv_vertices, axis=0).astype(np.float64)
    min_x, min_y = float(bounds_min[0]), float(bounds_min[1])
    span_x = max(float(bounds_max[0] - bounds_min[0]), 1e-9)
    span_y = max(float(bounds_max[1] - bounds_min[1]), 1e-9)
    texture_arr = bake_atlas(
        uv_vertices, new_faces, uvs, source, resolution, resolution,
        min_x, min_y, span_x, span_y
    )
    texture = Image.fromarray(texture_arr, mode="RGBA")
    texture.save(output_png)

    textured = trimesh.Trimesh(vertices=uv_vertices, faces=new_faces, process=False)
    textured.visual = trimesh.visual.texture.TextureVisuals(uv=uvs, image=texture)
    textured.export(output_glb, file_type="glb")

    # تحقق إعادة فتح GLB وأنه يحتوي هندسة وUV وخامة قابلة للقراءة.
    check = trimesh.load(output_glb, force="mesh", process=False)
    if len(check.vertices) == 0 or len(check.faces) == 0:
        raise RuntimeError("ملف GLB الناتج لا يحتوي هندسة.")
    visual = check.visual
    uv_check = getattr(visual, "uv", None)
    material = getattr(visual, "material", None)
    if uv_check is None or len(uv_check) != len(check.vertices):
        raise RuntimeError("فشل التحقق من UV coordinates في GLB الناتج.")
    if material is None:
        raise RuntimeError("لم يتم العثور على مادة Texture في GLB الناتج.")
    return {
        "vertices_after_unwrap": int(len(uv_vertices)),
        "faces": int(len(new_faces)),
        "texture_resolution": f"{resolution}x{resolution}",
        "glb_bytes": int(Path(output_glb).stat().st_size),
        "texture_bytes": int(Path(output_png).stat().st_size),
        "uv_verified": True,
        "texture_verified": True
    }

print("5/8 — إنشاء UV atlas وخامات للعينات الثلاث...")
results = []
for n, (image_path, mesh_path) in enumerate(samples, 1):
    stem = re.sub(r"[^A-Za-z0-9_-]+", "_", mesh_path.stem)
    glb_out = OUT / f"uv_{n:02d}_{stem}.glb"
    png_out = OUT / f"uv_{n:02d}_{stem}_texture.png"
    info = create_uv_texture(mesh_path, image_path, glb_out, png_out)
    row = {
        "sample": n,
        "source_image": str(image_path.relative_to(IMAGE_ROOT)),
        "source_glb": mesh_path.name,
        "output_glb": glb_out.name,
        "texture_png": png_out.name,
        **info,
        "status": "SUCCESS"
    }
    results.append(row)
    print(f"[{n}/3] SUCCESS — {glb_out.name} | {info['vertices_after_unwrap']:,} vertices | {info['glb_bytes']:,} bytes")

print("6/8 — إنشاء التقرير والبيانات الوصفية وملف ZIP...")
report_path = OUT / "uv_texture_report.csv"
with report_path.open("w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=list(results[0].keys()))
    writer.writeheader()
    writer.writerows(results)

metadata = {
    "project": "Conquer Mobile Assets",
    "stage": "3-model UV texture projection test",
    "method": "xatlas UV unwrap + image projection bake into RGBA UV atlas",
    "warning": "Single-image projection test only; it is not multi-view semantic texturing or a final production PBR material.",
    "source_archive": MODEL_ZIP.name,
    "matched_pairs": len(pairs),
    "results": results
}
(OUT / "metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")

zip_base = "/content/Conquer_UV_Texture_Test_3_Samples"
zip_path = Path(shutil.make_archive(zip_base, "zip", root_dir=WORK, base_dir="uv_texture_test"))
drive_copy = DRIVE / zip_path.name
shutil.copy2(zip_path, drive_copy)
print("نسخة Drive:", drive_copy)

print("7/8 — رفع الملفات إلى مجلد جديد على GitHub...")
try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
if not token:
    raise RuntimeError("تم إنشاء النتائج محليًا ونسخ ZIP إلى Drive، لكن Secret باسم GITHUB_TOKEN غير متاح. أضفه في Colab Secrets مع Contents: Read and write وفعّل Notebook access، ثم أعد تشغيل الخلية.")

api = f"https://api.github.com/repos/{REPO}/contents"
headers = {
    "Authorization": f"Bearer {token}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28"
}
uploads = [(p, f"uv_texture_test/{p.name}") for p in sorted(OUT.iterdir()) if p.is_file()]
uploads.append((zip_path, "uv_texture_test/Conquer_UV_Texture_Test_3_Samples.zip"))
uploaded = []
for local, remote in uploads:
    b64 = base64.b64encode(local.read_bytes()).decode("ascii")
    old = requests.get(f"{api}/{remote}", headers=headers, timeout=60)
    if old.status_code == 200:
        sha = old.json()["sha"]
    elif old.status_code == 404:
        sha = None
    else:
        old.raise_for_status()
    payload = {"message": f"Add UV texture test output: {Path(remote).name}", "content": b64, "branch": "main"}
    if sha:
        payload["sha"] = sha
    response = requests.put(f"{api}/{remote}", headers=headers, json=payload, timeout=180)
    response.raise_for_status()
    uploaded.append(remote)
    print("UPLOADED:", remote)

print("8/8 — التحقق من كل ملف مرفوع...")
for remote in uploaded:
    check = requests.get(f"{api}/{remote}", headers=headers, timeout=60)
    check.raise_for_status()
    print("VERIFIED:", remote)

print("\nاكتمل اختبار UV للعينات الثلاث.")
print("GitHub: https://github.com/" + REPO + "/tree/main/uv_texture_test")
print("Google Drive:", drive_copy)
print("مهم: هذه خامة إسقاط من صورة واحدة على كامل السطح؛ راجع العينات بصريًا قبل تعميمها على 42 مجسمًا. لا يوجد تعديل على الأصول الأصلية.")
